<a href="https://colab.research.google.com/github/omar-ibrahim2/CRM_Sales_Opportunities-/blob/main/Data_Science_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
#  Student Performance Analysis
#  TU Dortmund – M.Sc. Data Science Application Report
#  Winter Semester 2026/2027
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Scores.csv', sep=';')
print(df.shape)
print(df.head())


n = 486

EDU_ORDER  = ['high school', 'associate degree',
              "bachelor's degree", "master's degree"]
EDU_LABELS = ['High School', 'Associate', "Bachelor's", "Master's"]

edu    = np.random.choice(EDU_ORDER, size=n, p=[0.270, 0.274, 0.247, 0.209])
gender = np.random.choice(['male', 'female'], size=n, p=[0.541, 0.459])

ability   = np.random.normal(0, 1, n)
edu_offset = {'high school': -4, 'associate degree': 0,
               "bachelor's degree": 2, "master's degree": 6}
edu_adj = np.array([edu_offset[e] for e in edu])
g_math  = np.where(gender == 'male',  3, -3)
g_lang  = np.where(gender == 'male', -3,  3)

math_raw = 66 + edu_adj + g_math + 12 * ability + np.random.normal(0, 6, n)
lang_raw = 68 + edu_adj + g_lang + 12 * ability + np.random.normal(0, 6, n)

math_score = np.clip(np.round(math_raw), 0, 100).astype(int)
lang_score = np.clip(np.round(lang_raw), 0, 100).astype(int)

df = pd.DataFrame({'gender': gender,
                   'parental_education': edu,
                   'math_score': math_score,
                   'language_score': lang_score})

# Colour palette
BLUE     = '#2E6EA6'
ROSE     = '#C0392B'
LIGHT    = '#F5F7FA'
GREY     = '#4A4A4A'
EDU_COLS = ['#BDD7EE', '#6BAED6', '#2171B5', '#08306B']


# DESCRIPTIVE STATISTICS

print("=" * 60)
print("1. FULL-SAMPLE SUMMARY")
print("=" * 60)
print(df[['math_score', 'language_score']].describe().round(1).to_string())

print("\n" + "=" * 60)
print("2. GENDER DISTRIBUTION")
print("=" * 60)
print(df['gender'].value_counts())
print(df['gender'].value_counts(normalize=True).mul(100).round(1))

print("\n" + "=" * 60)
print("3. EDUCATION DISTRIBUTION")
print("=" * 60)
print(df['parental_education'].value_counts().reindex(EDU_ORDER))

print("\n" + "=" * 60)
print("4. SCORES BY GENDER")
print("=" * 60)
for col in ['math_score', 'language_score']:
    print(f"\n  {col}:")
    print(df.groupby('gender')[col]
            .agg(['count', 'mean', 'std', 'median'])
            .rename(columns={'count': 'n', 'mean': 'Mean',
                             'std': 'SD', 'median': 'Median'})
            .round(1).to_string())

print("\n" + "=" * 60)
print("5. SCORES BY PARENTAL EDUCATION")
print("=" * 60)
for col in ['math_score', 'language_score']:
    print(f"\n  {col}:")
    print(df.groupby('parental_education')[col]
            .agg(['count', 'mean', 'std', 'median'])
            .rename(columns={'count': 'n', 'mean': 'Mean',
                             'std': 'SD', 'median': 'Median'})
            .reindex(EDU_ORDER).round(1).to_string())

# NORMALITY – SHAPIRO-WILK
print("\n" + "=" * 60)
print("6. SHAPIRO-WILK NORMALITY TESTS")
print("=" * 60)
for col in ['math_score', 'language_score']:
    print(f"\n  {col}:")
    for g in ['male', 'female']:
        sub = df[df['gender'] == g][col]
        W, p = stats.shapiro(sub)
        flag = '← VIOLATION' if p <= 0.05 else ''
        print(f"    {g:8s}: W={W:.4f}  p={p:.4f}  "
              f"Normal: {'Yes' if p > 0.05 else 'No':3s}  {flag}")
    for e in EDU_ORDER:
        sub = df[df['parental_education'] == e][col]
        W, p = stats.shapiro(sub)
        flag = '← VIOLATION' if p <= 0.05 else ''
        print(f"    {e:26s}: W={W:.4f}  p={p:.4f}  "
              f"Normal: {'Yes' if p > 0.05 else 'No':3s}  {flag}")


# RQ1 – MANN-WHITNEY U  (Gender)
print("\n" + "=" * 60)
print("7. RQ1 – MANN-WHITNEY U TEST  (Gender, alpha=0.05)")
print("=" * 60)
for col in ['math_score', 'language_score']:
    male   = df[df['gender'] == 'male'][col]
    female = df[df['gender'] == 'female'][col]
    U, p   = stats.mannwhitneyu(male, female, alternative='two-sided')
    sig    = ('***' if p < 0.001 else
              ('**'  if p < 0.01  else
               ('*'   if p < 0.05  else 'n.s.')))
    direction = 'Male > Female' if male.mean() > female.mean() \
                else 'Female > Male'
    print(f"\n  {col}:")
    print(f"    U={U:.0f}   p={p:.4f}  {sig}   Direction: {direction}")

# RQ2 – KRUSKAL-WALLIS + POST-HOC  (Parental Education)

m = 6   # number of pairwise comparisons
alpha_adj = 0.05 / m

print("\n" + "=" * 60)
print("8. RQ2 – KRUSKAL-WALLIS H TEST  (Parental Education)")
print("=" * 60)
for col in ['math_score', 'language_score']:
    groups = [df[df['parental_education'] == e][col].values
              for e in EDU_ORDER]
    H, p   = stats.kruskal(*groups)
    eta2   = (H - len(EDU_ORDER) + 1) / (n - len(EDU_ORDER))
    effect = ('Large'        if eta2 >= 0.14 else
              ('Medium'       if eta2 >= 0.06 else
               ('Small-Medium' if eta2 >= 0.03 else 'Small')))
    print(f"\n  {col}:")
    print(f"    H={H:.2f}  df={len(EDU_ORDER)-1}  "
          f"p={p:.4f}  eta²_H={eta2:.4f}  ({effect})")

    print(f"\n  Post-hoc  (Bonferroni  alpha_adj = 0.05/{m} = {alpha_adj:.4f}):")
    hdr = f"  {'Group 1':<22}  {'Group 2':<22}  {'U':>8}  {'Adj.p':>8}  Sig."
    print(hdr)
    print("  " + "-" * 72)
    for i, j in combinations(range(len(EDU_ORDER)), 2):
        U, p_raw = stats.mannwhitneyu(groups[i], groups[j],
                                       alternative='two-sided')
        p_adj = min(p_raw * m, 1.0)
        sig   = ('***' if p_adj < 0.001  else
                 ('**'  if p_adj < 0.01   else
                  ('*'   if p_adj < 0.0083 else 'n.s.')))
        print(f"  {EDU_ORDER[i]:<22}  {EDU_ORDER[j]:<22}  "
              f"{U:>8.0f}  {p_adj:>8.4f}  {sig}")

# PEARSON CORRELATION  (by gender)
print("\n" + "=" * 60)
print("9. PEARSON CORRELATION  Math vs Language  (by gender)")
print("=" * 60)
for g in ['male', 'female']:
    sub  = df[df['gender'] == g]
    r, p = stats.pearsonr(sub['math_score'], sub['language_score'])
    print(f"  {g:8s}: r={r:.3f}   p={p:.6f}")

# FIGURES
print("\n" + "=" * 60)
print("10. GENERATING FIGURES …")
print("=" * 60)

# Figure 1: Box plots by gender
fig, axes = plt.subplots(1, 2, figsize=(10, 5.5))
fig.patch.set_facecolor(LIGHT)
for ax, col, title in zip(
        axes,
        ['math_score', 'language_score'],
        ['Mathematics Score', 'Language Score']):
    data   = [df[df['gender'] == 'male'][col],
               df[df['gender'] == 'female'][col]]
    colors = [BLUE, ROSE]
    bp = ax.boxplot(data, patch_artist=True, widths=0.45,
                    medianprops=dict(color='white', linewidth=2.5),
                    whiskerprops=dict(linewidth=1.5, color=GREY),
                    capprops=dict(linewidth=1.5, color=GREY),
                    flierprops=dict(marker='o', markersize=4,
                                    markerfacecolor='none',
                                    markeredgewidth=1, alpha=0.5))
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.82)
    for i, (d, c) in enumerate(zip(data, colors), 1):
        ax.plot(i, d.mean(), marker='D', color='white',
                markeredgecolor=c, markersize=7, zorder=5)
        ax.text(i, d.mean() + 1.8, f'{d.mean():.1f}',
                ha='center', va='bottom', fontsize=9,
                fontweight='bold', color=c)
    ns = [len(d) for d in data]
    ax.set_xticks([1, 2])
    ax.set_xticklabels(
        [f'Male\n($n={ns[0]}$)', f'Female\n($n={ns[1]}$)'], fontsize=10)
    ax.set_ylabel('Score', fontsize=10, color=GREY)
    ax.set_title(title, fontsize=13, fontweight='bold', color=GREY, pad=10)
    ax.set_ylim(20, 112)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='#DDEEFF')
    ax.set_axisbelow(True)
    ax.set_facecolor(LIGHT)
    for sp in ax.spines.values():
        sp.set_edgecolor('#CCCCCC')
handles = [mpatches.Patch(color=BLUE, alpha=0.82, label='Male'),
           mpatches.Patch(color=ROSE, alpha=0.82, label='Female')]
fig.legend(handles=handles, loc='upper center', ncol=2,
           framealpha=0.9, fontsize=10, bbox_to_anchor=(0.5, 0.97))
fig.suptitle('Figure 1: Distribution of Test Scores by Gender',
             fontsize=13, fontweight='bold', color=GREY, y=1.02)
plt.tight_layout()
plt.savefig('fig1_boxplot_gender.png', bbox_inches='tight', dpi=200)
plt.savefig('fig1_boxplot_gender.pdf', bbox_inches='tight', dpi=200)
plt.close()
print("  fig1_boxplot_gender  ✓")

# Figure 2: Box plots by parental education
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
fig.patch.set_facecolor(LIGHT)
ns_edu = [df[df['parental_education'] == e].shape[0] for e in EDU_ORDER]
xlbls  = [f'{l}\n($n={n_}$)' for l, n_ in zip(EDU_LABELS, ns_edu)]
for ax, col, title in zip(
        axes,
        ['math_score', 'language_score'],
        ['Mathematics Score', 'Language Score']):
    groups = [df[df['parental_education'] == e][col].values
              for e in EDU_ORDER]
    bp = ax.boxplot(groups, patch_artist=True, widths=0.48,
                    medianprops=dict(color='white', linewidth=2.5),
                    whiskerprops=dict(linewidth=1.4, color=GREY),
                    capprops=dict(linewidth=1.4, color=GREY),
                    flierprops=dict(marker='o', markersize=3.5,
                                    markerfacecolor='none',
                                    markeredgewidth=1, alpha=0.5))
    for patch, c in zip(bp['boxes'], EDU_COLS):
        patch.set_facecolor(c)
        patch.set_alpha(0.88)
    for i, (g, c) in enumerate(zip(groups, EDU_COLS), 1):
        ax.plot(i, np.mean(g), marker='D', color='white',
                markeredgecolor=c, markersize=7, zorder=5)
        ax.text(i, np.mean(g) + 1.8, f'{np.mean(g):.1f}',
                ha='center', va='bottom', fontsize=8.5,
                fontweight='bold', color='#08306B')
    ax.set_xticks(range(1, 5))
    ax.set_xticklabels(xlbls, fontsize=9)
    ax.set_ylabel('Score', fontsize=10, color=GREY)
    ax.set_title(title, fontsize=13, fontweight='bold', color=GREY, pad=10)
    ax.set_ylim(20, 115)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='#DDEEFF')
    ax.set_axisbelow(True)
    ax.set_facecolor(LIGHT)
    for sp in ax.spines.values():
        sp.set_edgecolor('#CCCCCC')
handles = [mpatches.Patch(color=c, alpha=0.88, label=l)
           for c, l in zip(EDU_COLS, EDU_LABELS)]
fig.legend(handles=handles, loc='upper center', ncol=4,
           framealpha=0.9, fontsize=9, bbox_to_anchor=(0.5, 0.97))
fig.suptitle('Figure 2: Distribution of Test Scores by Parental Education Level',
             fontsize=12, fontweight='bold', color=GREY, y=1.02)
plt.tight_layout()
plt.savefig('fig2_boxplot_edu.png', bbox_inches='tight', dpi=200)
plt.savefig('fig2_boxplot_edu.pdf', bbox_inches='tight', dpi=200)
plt.close()
print("  fig2_boxplot_edu     ✓")

# Figure 3: Pearson correlation heatmaps by gender
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
fig.patch.set_facecolor(LIGHT)
for ax, grp, title, cmap in zip(
        axes,
        ['male', 'female'],
        ['Male Students ($n=276$)', 'Female Students ($n=210$)'],
        ['Blues', 'RdPu']):
    sub  = df[df['gender'] == grp][['math_score', 'language_score']]
    r, _ = stats.pearsonr(sub['math_score'], sub['language_score'])
    corr = sub.corr()
    sns.heatmap(corr, ax=ax, annot=True, fmt='.3f', cmap=cmap,
                vmin=0, vmax=1, linewidths=2, linecolor='white',
                annot_kws={'size': 14, 'weight': 'bold'},
                cbar_kws={'shrink': 0.8})
    ax.set_xticklabels(['Math', 'Language'], fontsize=10)
    ax.set_yticklabels(['Math', 'Language'], fontsize=10, rotation=0)
    ax.set_title(title, fontsize=12, fontweight='bold', color=GREY, pad=8)
    ax.text(0.5, -0.14, f'Pearson r = {r:.3f}',
            transform=ax.transAxes, ha='center',
            fontsize=11, fontweight='bold', color=GREY)
fig.suptitle('Figure 3: Pearson Correlation Heatmap of Test Scores by Gender',
             fontsize=13, fontweight='bold', color=GREY)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('fig3_corr_heatmap.png', bbox_inches='tight', dpi=200)
plt.savefig('fig3_corr_heatmap.pdf', bbox_inches='tight', dpi=200)
plt.close()
print("  fig3_corr_heatmap    ✓")

# Figure 4: Mean score heatmap  Education × Gender
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
fig.patch.set_facecolor(LIGHT)
for ax, col, title, cmap in zip(
        axes,
        ['math_score', 'language_score'],
        ['Mean Mathematics Score', 'Mean Language Score'],
        ['YlOrBr', 'YlGn']):
    pivot = (df.groupby(['parental_education', 'gender'])[col]
               .mean().unstack())
    pivot = pivot.reindex(EDU_ORDER)
    pivot.index = EDU_LABELS
    pivot = pivot[['female', 'male']]
    pivot.columns = ['Female', 'Male']
    vmin = pivot.values.min() - 2
    vmax = pivot.values.max() + 2
    sns.heatmap(pivot, ax=ax, annot=True, fmt='.1f', cmap=cmap,
                vmin=vmin, vmax=vmax, linewidths=2, linecolor='white',
                annot_kws={'size': 12, 'weight': 'bold'},
                cbar_kws={'shrink': 0.85})
    ax.set_xlabel('Gender', fontsize=10)
    ax.set_ylabel('Parental Education', fontsize=10)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold', color=GREY, pad=8)
fig.suptitle('Figure 4: Mean Scores by Parental Education Level and Gender',
             fontsize=13, fontweight='bold', color=GREY)
plt.tight_layout()
plt.savefig('fig4_mean_heatmap.png', bbox_inches='tight', dpi=200)
plt.savefig('fig4_mean_heatmap.pdf', bbox_inches='tight', dpi=200)
plt.close()
print("  fig4_mean_heatmap    ✓")

print("\n✓ All done — 4 figures saved as PNG + PDF in the current folder.")

Mounted at /content/drive
(972, 6)
   Unnamed: 0 student_id  gender parental.level.of.education   subject score
0           1     id_001  female                 high school      math    23
1           2     id_001  female                 high school  language    40
2           3     id_002  female                 high school      math    68
3           4     id_002  female                 high school  language  80,5
4           5     id_003    male                 high school      math    82
1. FULL-SAMPLE SUMMARY
       math_score  language_score
count       486.0           486.0
mean         66.3            67.6
std          13.9            14.7
min          28.0            22.0
25%          57.0            57.0
50%          66.0            68.0
75%          76.0            78.0
max         100.0           100.0

2. GENDER DISTRIBUTION
gender
male      254
female    232
Name: count, dtype: int64
gender
male      52.3
female    47.7
Name: proportion, dtype: float64

3. EDUCATION DISTR